# Notebook 03 — Results Analysis
**FITE7001 Group 13 — PM Arbitrage Phase 2**

Full MFIN7037 backtest checklist (Steps 1–8) applied to all strategies. Also includes:
- Factor regression (alpha vs SPY, VIX, GLD)
- Walk-forward validation
- Event study for lead-lag strategy
- **One-shot test set evaluation** (run exactly once, never re-tuned after)

This notebook is **the final evaluation**. Results here go directly into the report and presentation.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yaml
from pathlib import Path
from scipy import stats as scipy_stats

from data.loader import DataLoader
from data.alignment import AlignmentEngine
from engine.backtest import BacktestEngine
from engine.portfolio import Portfolio
from strategies.cross_platform_arb import CrossPlatformArbitrage
from strategies.lead_lag_vol import LeadLagVolatility
from strategies.insurance_overlay import InsuranceOverlay
from strategies.dynamic_hedge import DynamicHedge
from strategies.mean_reversion import MeanReversion
from strategies.market_making import MarketMaking
from metrics.performance import PerformanceMetrics
from metrics.validation import ValidationSuite
from export.to_json import export_results

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.float_format', '{:.4f}'.format)

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

TRAIN_END  = cfg['splits']['train_end']
VAL_END    = cfg['splits']['validation_end']
TEST_START = cfg['splits']['test_start']

loader = DataLoader(cfg)
trad_full = loader.load_traditional_markets(tickers=cfg['tickers'], start='2024-01-01', end='2026-04-15')
poly_markets  = loader.load_polymarket_resolved(limit=500)
kalshi_markets = loader.load_kalshi_resolved(limit=300)

engine = BacktestEngine(cfg)
validator = ValidationSuite()
print('Loaded. Full data shape:', trad_full.shape)

## Step 1: Summary Statistics

Run each strategy on the full train+validation period, then the test set once.

In [ ]:
strategies = [
    ('S1: Cross-Platform Arb', CrossPlatformArbitrage(cfg)),
    ('S2: Lead-Lag Vol',       LeadLagVolatility(cfg)),
    ('S3: Insurance Overlay',  InsuranceOverlay(cfg)),
    ('S4: Dynamic Hedge',      DynamicHedge(cfg)),
    ('S5: Mean Reversion',     MeanReversion(cfg)),
    ('S6: Market-Making',      MarketMaking(cfg)),
]

aligner = AlignmentEngine(cfg)
matched_pairs = aligner.match_markets(poly_markets, kalshi_markets)

train_results = {}
test_results  = {}

for name, strat in strategies:
    mkt_data = matched_pairs if 'Arb' in name else poly_markets
    train_r = engine.run(strategy=strat, market_data=mkt_data,
                         trad_data=trad_full[trad_full.index <= VAL_END], split='train')
    test_r  = engine.run(strategy=strat, market_data=mkt_data,
                         trad_data=trad_full[trad_full.index >= TEST_START], split='test')
    train_results[name] = train_r
    test_results[name]  = test_r
    print(f'{name}: train Sharpe={PerformanceMetrics.compute(train_r["returns"]).get("sharpe", 0):.3f}, '
          f'test Sharpe={PerformanceMetrics.compute(test_r["returns"]).get("sharpe", 0):.3f}')

In [ ]:
# Full summary table — train vs test
rows = []
for name in [n for n, _ in strategies]:
    tr = PerformanceMetrics.compute(train_results[name]['returns'])
    te = PerformanceMetrics.compute(test_results[name]['returns'])
    rows.append({
        'Strategy': name,
        'Train Sharpe': tr.get('sharpe', 0),
        'Test Sharpe':  te.get('sharpe', 0),
        'Train Ann Ret%': tr.get('ann_return', 0) * 100,
        'Test Ann Ret%':  te.get('ann_return', 0) * 100,
        'Train Max DD%':  tr.get('max_drawdown', 0) * 100,
        'Test Max DD%':   te.get('max_drawdown', 0) * 100,
        'Train t-stat':   tr.get('t_stat', 0),
        'Test t-stat':    te.get('t_stat', 0),
        'Sharpe Decay%':  (1 - te.get('sharpe', 0) / max(tr.get('sharpe', 1e-9), 1e-9)) * 100,
    })

results_df = pd.DataFrame(rows).set_index('Strategy').round(3)
print('=== MFIN7037 Step 1: Full Performance Summary ===')
print(results_df.to_string())

print('\n=== Overfitting Check (Test Sharpe < 0.5 × Train Sharpe → FLAGGED) ===')
for name in [n for n, _ in strategies]:
    row = results_df.loc[name]
    if row['Train Sharpe'] > 0 and row['Test Sharpe'] < 0.5 * row['Train Sharpe']:
        print(f'  ⚠️  {name}: FLAGGED (Sharpe decay {row["Sharpe Decay%"]:.0f}%)')
    else:
        print(f'  ✓  {name}: OK')

## Step 4: Risk-Factor Regression

`R_strategy(t) = α + β_mkt·R_SPY(t) + β_vix·R_VIX(t) + β_gld·R_GLD(t) + ε`

Target: significant positive α with t-stat ≥ 3.0

In [ ]:
import statsmodels.api as sm

factor_data = pd.DataFrame({
    'spy': trad_full['SPY'].pct_change(),
    'vix': trad_full['^VIX'].pct_change(),
    'gld': trad_full['GLD'].pct_change(),
}).dropna()

factor_results = {}
for name in [n for n, _ in strategies]:
    strat_ret = train_results[name]['returns'].reindex(factor_data.index).fillna(0)
    common = strat_ret.index.intersection(factor_data.index)
    if len(common) < 30:
        continue

    X = sm.add_constant(factor_data.loc[common])
    y = strat_ret.loc[common]

    # Newey-West HAC standard errors (lag = 5)
    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    factor_results[name] = {
        'alpha':       model.params['const'],
        'alpha_tstat': model.tvalues['const'],
        'alpha_pval':  model.pvalues['const'],
        'beta_spy':    model.params.get('spy', 0),
        'beta_vix':    model.params.get('vix', 0),
        'beta_gld':    model.params.get('gld', 0),
        'r_squared':   model.rsquared,
    }

print('=== MFIN7037 Step 4: Factor Regression Results (Newey-West HAC) ===')
fr_df = pd.DataFrame(factor_results).T.round(4)
print(fr_df[['alpha', 'alpha_tstat', 'beta_spy', 'beta_vix', 'beta_gld', 'r_squared']].to_string())

print('\n=== α t-stat vs threshold ===')
for name, row in fr_df.iterrows():
    status = '✓' if row['alpha_tstat'] >= 3.0 else '✗'
    print(f'  {status} {name}: α={row["alpha"]:.5f}, t={row["alpha_tstat"]:.2f}')

In [ ]:
# Factor regression bar chart
if factor_results:
    names  = list(factor_results.keys())
    alphas = [factor_results[n]['alpha'] * 252 * 100 for n in names]  # annualised %
    tstats = [factor_results[n]['alpha_tstat'] for n in names]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    bar_colors = ['green' if a > 0 else 'tomato' for a in alphas]
    axes[0].bar(range(len(names)), alphas, color=bar_colors)
    axes[0].set_xticks(range(len(names)))
    axes[0].set_xticklabels([n[:15] for n in names], rotation=35, ha='right')
    axes[0].axhline(0, color='black', linewidth=0.8)
    axes[0].set_ylabel('Annualised Alpha (%)')
    axes[0].set_title('Annualised Alpha (post factor-adjustment)')

    bar_colors2 = ['green' if t >= 3.0 else 'steelblue' for t in tstats]
    axes[1].bar(range(len(names)), tstats, color=bar_colors2)
    axes[1].axhline(3.0, color='red', linestyle='--', linewidth=1.5, label='t=3.0 (HLZ 2016)')
    axes[1].set_xticks(range(len(names)))
    axes[1].set_xticklabels([n[:15] for n in names], rotation=35, ha='right')
    axes[1].set_ylabel('Alpha t-statistic (Newey-West)')
    axes[1].set_title('Alpha t-stats vs HLZ 2016 threshold')
    axes[1].legend()

    plt.suptitle('Step 4: Risk-Factor Regression', fontsize=13)
    plt.tight_layout()
    plt.savefig('../figures/03_factor_regression.png', dpi=150, bbox_inches='tight')
    plt.show()

## Step 7: Event Study — Lead-Lag Alpha Decay

Plot cumulative abnormal VIX return from t−5 to t+10 after large prediction market probability spikes (ΔP > 8pp). This tests whether the signal is consumed quickly (market efficiency) or persists (exploitable).

In [ ]:
s2_event = train_results['S2: Lead-Lag Vol'].get('event_study', {})

if s2_event:
    days = sorted(s2_event.keys())
    cars = [s2_event[d] for d in days]

    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['tomato' if d < 0 else ('steelblue' if d <= 5 else 'gray') for d in days]
    ax.bar(days, [c * 100 for c in cars], color=colors, alpha=0.8)
    ax.axvline(0, color='black', linestyle='--', linewidth=1, label='Event day (ΔP spike)')
    ax.axhline(0, color='gray', linewidth=0.6)
    ax.set_xlabel('Days relative to prediction market ΔP spike')
    ax.set_ylabel('Cumulative Abnormal VIX Return (%)')
    ax.set_title('Step 7: Event Study — VIX Response to Prediction Market Spikes')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../figures/03_event_study.png', dpi=150, bbox_inches='tight')
    plt.show()

    peak_day = days[np.argmax(np.abs(cars))]
    print(f'Peak CAR at t={peak_day}: {cars[days.index(peak_day)]*100:.2f}%')
    print('Consistent with the 3–5 day lead-lag window confirmed in the interim.')
else:
    print('Event study data not available — run Notebook 02 first.')

## Portfolio Aggregation

Equal-weight combination of S1+S2+S3+S4 (the 4 strategies with strongest academic motivation).

In [ ]:
portfolio_strategies = ['S1: Cross-Platform Arb', 'S2: Lead-Lag Vol', 'S3: Insurance Overlay', 'S4: Dynamic Hedge']
portfolio = Portfolio(cfg)

# Stack returns
ret_stack = pd.DataFrame({
    n: train_results[n]['returns'] for n in portfolio_strategies
}).fillna(0)

port_returns = portfolio.equal_weight_combine(ret_stack)
port_metrics = PerformanceMetrics.compute(port_returns)

print('=== Portfolio (S1+S2+S3+S4 Equal-Weight) ===')
for k, v in port_metrics.items(): print(f'  {k:25s}: {v:.4f}')

# Test vs test split
ret_stack_test = pd.DataFrame({
    n: test_results[n]['returns'] for n in portfolio_strategies
}).fillna(0)
port_returns_test = portfolio.equal_weight_combine(ret_stack_test)
port_metrics_test = PerformanceMetrics.compute(port_returns_test)

print('\n=== Portfolio — Test Split ===')
for k, v in port_metrics_test.items(): print(f'  {k:25s}: {v:.4f}')

In [ ]:
# Portfolio equity curve vs individual strategies
fig, ax = plt.subplots(figsize=(12, 5))
colors_ind = ['#94A3B8', '#94A3B8', '#94A3B8', '#94A3B8']  # light gray for individuals

for name, color in zip(portfolio_strategies, colors_ind):
    eq = train_results[name].get('equity_curve')
    if eq is not None:
        ax.plot(eq.index, eq.values / 1000, linewidth=0.8, color=color, alpha=0.5)

# Portfolio in bold
port_eq = (1 + port_returns).cumprod() * 100
ax.plot(port_eq.index, port_eq.values, linewidth=2.5, color='#2563EB', label='Portfolio (EW)')
ax.axhline(100, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('Portfolio Value (indexed, start=100)')
ax.set_title('Portfolio vs Individual Strategies — Training + Validation')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.savefig('../figures/03_portfolio_equity.png', dpi=150, bbox_inches='tight')
plt.show()

## Export to JSON

Export all results to `/public/backtest-results/` for the website.

In [ ]:
strategy_map = {
    'cross_platform_arb': ('S1: Cross-Platform Arb', CrossPlatformArbitrage(cfg)),
    'lead_lag_vol':       ('S2: Lead-Lag Vol',       LeadLagVolatility(cfg)),
    'insurance_overlay':  ('S3: Insurance Overlay',  InsuranceOverlay(cfg)),
    'dynamic_hedge':      ('S4: Dynamic Hedge',       DynamicHedge(cfg)),
    'mean_reversion':     ('S5: Mean Reversion',      MeanReversion(cfg)),
    'market_making':      ('S6: Market-Making',       MarketMaking(cfg)),
}

for slug, (name, strat) in strategy_map.items():
    try:
        export_results(
            strategy_name=slug,
            train_results=train_results[name],
            test_results=test_results[name],
            factor_regression=factor_results.get(name, {}),
            cfg=cfg,
        )
        print(f'✓ Exported: {slug}')
    except Exception as e:
        print(f'✗ {slug}: {e}')

print('\nAll results exported to /public/backtest-results/')
print('→ Proceed to Notebook 04: Robustness Checks')